<a href="https://colab.research.google.com/github/yasaswini1408/Campus_Connect/blob/main/Exp_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain-tavily langchain-groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.8 MB/s eta 0:00:00


In [2]:
import os
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_tavily import TavilySearch
from langchain_groq import ChatGroq
from google.colab import userdata
from pprint import pprint

In [4]:
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')

In [8]:
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.1
    )
@tool
def get_weather(location: str) -> str:
  """Get weather information for a location."""
  return f"Weather in {location}: Sunny, 72°F"
  # Initialize our external search tool
  web_search_tool = TavilySearch(max_results=2)
  tools_list = [web_search_tool, get_weather]

In [11]:
system_prompt = (
    "You are a helpful, precise AI assistant equipped with external tools.\n\n"
    "Your goal is to answer the user's requests accurately. You have access "
    "to tools that can fetch real-time data, search the web, or check the "
    "weather.\n\n"
    "Guidelines:\n"
    "1. Always evaluate if the user's request requires real-time information "
    "or specific knowledge you don't possess. If it does, call the "
    "appropriate tool.\n"
    "2. If the tools return information, synthesize a clear, comprehensive, "
    "and factual response for the user based on those results.\n"
    "3. Be honest about what you know and don't know. If the tools do not "
    "provide an answer, state that clearly.\n"
    )
web_search_tool = TavilySearch(max_results=2)
tools_list = [web_search_tool, get_weather]

agent = create_agent(model, tools=tools_list, system_prompt=system_prompt)

In [12]:
print("--- Launching Agent via Direct Invoke ---")
# We request recent data that forces the agent to use its search tool
query = input("Enter your query: ")
# Execute the agent workflow directly without an external wrapper
response = agent.invoke(
    {"messages": [{"role": "human", "content": f"{query}"}]}
)
print("---FINAL RESPONSE---")
pprint(response)
print(response["messages"][-1].content)

--- Launching Agent via Direct Invoke ---
Enter your query: Is there any IPL cricket match on May 26th 2026 
---FINAL RESPONSE---
{'messages': [HumanMessage(content='Is there any IPL cricket match on May 26th 2026 ', additional_kwargs={}, response_metadata={}, id='ee699374-ac60-4702-92a7-ccb8d76c241d'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Is there any IPL cricket match on May 26th 2026". Need to check schedule of IPL 2026. Likely not known. Need to search web for IPL 2026 schedule. Use search tool.', 'tool_calls': [{'id': 'fc_13b69bff-b189-4bdd-9ad2-e4c4035db52e', 'function': {'arguments': '{"query":"IPL 2026 schedule May 26 2026 match","search_depth":"advanced","time_range":"year"}', 'name': 'tavily_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 107, 'prompt_tokens': 479, 'total_tokens': 586, 'completion_time': 0.232914761, 'completion_tokens_details': {'reasoning_tokens': 51}, 'prompt_t